Install Libraries

In [1]:
!pip install -U transformers

In [2]:
!pip install transformers datasets torch scikit-learn pandas

Import Libraries

In [3]:
import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

from transformers import BertTokenizer, BertForSequenceClassification
from transformers import Trainer, TrainingArguments

Load Dataset

In [21]:
df = pd.read_csv("fake_job_postings.csv", on_bad_lines='skip', engine='python')

print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (3031, 18)


,job_id,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent
0,1,Marketing Intern,"US, NY, New York",Marketing,NaN,"We're Food52, and we've created a groundbreaki...","Food52, a fast-growing, James Beard Award-winn...",Experience with content management systems a m...,NaN,0,1,0,Other,Internship,NaN,NaN,Marketing,0
1,2,Customer Service - Cloud Video Production,"NZ, , Auckland",Success,NaN,"90 Seconds, the worlds Cloud Video Production ...",Organised - Focused - Vibrant - Awesome!Do you...,What we expect from you:Your key responsibilit...,What you will get from usThrough being part of...,0,1,0,Full-time,Not Applicable,NaN,Marketing and Advertising,Customer Service,0
2,3,Commissioning Machinery Assistant (CMA),"US, IA, Wever",NaN,NaN,Valor Services provides Workforce Solutions th...,"Our client, located in Houston, is actively se...",Implement pre-commissioning and commissioning ...,NaN,0,1,0,NaN,NaN,NaN,NaN,NaN,0
3,4,Account Executive - Washington DC,"US, DC, Washington",Sales,NaN,Our passion for improving quality of life thro...,THE COMPANY: ESRI – Environmental Systems Rese...,"EDUCATION: Bachelor’s or Master’s in GIS, busi...",Our culture is anything but corporate—we have ...,0,1,0,Full-time,Mid-Senior level,Bachelor's Degree,Computer Software,Sales,0
4,5,Bill Review Manager,"US, FL, Fort Worth",NaN,NaN,SpotSource Solutions LLC is a Global Human Cap...,JOB TITLE: Itemization Review ManagerLOCATION:...,QUALIFICATIONS:RN license in the State of Texa...,Full Benefits Offered,0,1,1,Full-time,Mid-Senior level,Bachelor's Degree,Hospital & Health Care,Health Care Provider,0


Handle Missing Values

In [5]:
text_columns = [
    "title",
    "company_profile",
    "description",
    "requirements",
    "benefits"
]

for col in text_columns:
    df[col] = df[col].fillna("")

Combine Text Columns

In [6]:
df["text"] = (
    df["title"] + " " +
    df["company_profile"] + " " +
    df["description"] + " " +
    df["requirements"] + " " +
    df["benefits"]
)

X = df["text"]
y = df["fraudulent"]

Train Test Split

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train Size:", len(X_train))
print("Test Size:", len(X_test))

Train Size: 2424
Test Size: 607


Load BERT Tokenizer

In [8]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Tokenize Text

In [9]:
train_encodings = tokenizer(
    list(X_train),
    truncation=True,
    padding=True,
    max_length=512
)

test_encodings = tokenizer(
    list(X_test),
    truncation=True,
    padding=True,
    max_length=512
)

Create PyTorch Dataset

In [10]:
class JobDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.reset_index(drop=True)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)


train_dataset = JobDataset(train_encodings, y_train)
test_dataset = JobDataset(test_encodings, y_test)

Load BERT Model

In [11]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [12]:
from sklearn.utils.class_weight import compute_class_weight
import torch

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

class_weights = torch.tensor(class_weights, dtype=torch.float)
print(class_weights)

tensor([ 0.5157, 16.3784])


1.1Training Configuration

In [13]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",      # ✅ correct for v5.x
    save_strategy="epoch",
    logging_steps=100,
    load_best_model_at_end=True
)

compute_metrics

In [14]:
from sklearn.metrics import accuracy_score

def compute_metrics(pred):

    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc
    }

1.2 Train the Model

In [15]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.162916,0.099186,0.975288
2,0.079903,0.093475,0.980231
3,0.057199,0.087086,0.981878


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=456, training_loss=0.09059066469209236, metrics={'train_runtime': 891.0908, 'train_samples_per_second': 8.161, 'train_steps_per_second': 0.512, 'total_flos': 1913343594577920.0, 'train_loss': 0.09059066469209236, 'epoch': 3.0})

Evaluation Metrics

In [16]:
from sklearn.metrics import classification_report
import numpy as np

# Make predictions
predictions = trainer.predict(test_dataset)

# Convert probabilities to labels
y_pred = np.argmax(predictions.predictions, axis=1)

# Print classification metrics
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))


Classification Report:

              precision    recall  f1-score   support

           0       0.98      1.00      0.99       589
           1       0.89      0.44      0.59        18

    accuracy                           0.98       607
   macro avg       0.94      0.72      0.79       607
weighted avg       0.98      0.98      0.98       607



1.3 Evaluate Model

In [17]:
predictions = trainer.predict(test_dataset)

y_pred = np.argmax(predictions.predictions, axis=1)

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Accuracy: 0.9818780889621087

Classification Report:

              precision    recall  f1-score   support

           0       0.98      1.00      0.99       589
           1       0.89      0.44      0.59        18

    accuracy                           0.98       607
   macro avg       0.94      0.72      0.79       607
weighted avg       0.98      0.98      0.98       607



1.4 Prediction Function

In [18]:
def predict_job_bert(text):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model.to(device)

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )

    # move tensors to GPU
    inputs = {key: value.to(device) for key, value in inputs.items()}

    outputs = model(**inputs)

    probs = torch.nn.functional.softmax(outputs.logits, dim=1)

    fake_prob = probs[0][1].item()
    real_prob = probs[0][0].item()

    print("\nPrediction Result:\n")

    if fake_prob > real_prob:
        print("⚠ FAKE JOB")
        print("Scam Probability:", round(fake_prob*100,2), "%")
    else:
        print("✅ REAL JOB")
        print("Authenticity Probability:", round(real_prob*100,2), "%")

1.5 Test BERT Model

In [19]:
sample_job = """
Work from home. Earn 50,000 per week.
No experience required.
Registration fee of 2000 INR required.
"""

predict_job_bert(sample_job)


Prediction Result:

✅ REAL JOB
Authenticity Probability: 98.65 %


1.6 Save BERT Model

In [20]:
model.save_pretrained("bert_fake_job_model")
tokenizer.save_pretrained("bert_fake_job_model")

print("BERT Model Saved Successfully")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

BERT Model Saved Successfully
